# Importar Librerias

In [87]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Para entrenar modelo
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import shap
from sklearn.metrics import precision_score, recall_score, f1_score



import warnings
warnings.filterwarnings('ignore')

##### Cargar dataset

# Importar Librerias

In [88]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Para entrenar modelo
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import shap



import warnings
warnings.filterwarnings('ignore')

Variables globales requeridas y parametros

In [89]:
FEATURES = [
    # Usadas en el clustering
    'PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes',
    'has_extracurriculars', 'parents_exatec_enc', 'total.scholarship.loan', 'FTE',
    
    # Se agregan estas
    'admission.rubric',      # information gain relevante
    'english.evaluation',    # nivel de inglés, predictor académico
    'age',                   # info gain 0.0086 en el paper
    'online.test',           # ya es binario, sin problema
    
]
# El objetivo a predecir
TARGET = 'dropout'
COLS_TO_SCALE = ['PNA', 'admission.rubric', 'english.evaluation', 'age']
THRESHOLD = 0.5

##### Cargar dataset

In [90]:
def load_clusters(filepath):
    df = pd.read_csv(filepath)
    df[TARGET] = (df['retention'] == 0).astype(int)
    clusters = []
    for cluster_id in sorted(df['cluster_id'].unique()):
        cluster_df = df[df['cluster_id'] == cluster_id].copy()
        cluster_df = cluster_df[FEATURES + [TARGET]]
        clusters.append(cluster_df)
    return clusters

##### Obtener los clusters

In [91]:
def load_clusters(filepath):
    print("================== Cargando:" + filepath)
    df = pd.read_csv(filepath)
    # Crear columna dropout a partir de retention
    clusters = []
    for cluster_id in sorted(df['cluster_id'].unique()):
        cluster_df = df[df['cluster_id'] == cluster_id].copy()
        # Solo dejar columnas necesarias
        cluster_df = cluster_df[FEATURES + [TARGET]]
        clusters.append(cluster_df)
        print(f"Cluster {cluster_id}: {len(cluster_df)} filas")
    print(f"Total de clusters creados: {len(clusters)}")
    return clusters

Funcion para entrenar regresion logistica

In [92]:
def train_cluster_model(cluster_df, cluster_id):
    # Dropear columnas constantes (hay un solo valor unico) dinámicamente
    constant_cols = [col for col in cluster_df.columns
                     if cluster_df[col].nunique() <= 1 and col != TARGET]
    df_clean = cluster_df.drop(columns=constant_cols)

    if constant_cols:
        print(f"Cluster {cluster_id} — columnas constantes dropeadas: {constant_cols}")

    X = df_clean.drop(columns=[TARGET])
    y = df_clean[TARGET]

    # Escalar columnas numéricas (solo las que existen después del drop)
    cols_scale = [c for c in COLS_TO_SCALE if c in X.columns]
    scaler = StandardScaler()
    X[cols_scale] = scaler.fit_transform(X[cols_scale])

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Aplicar SMOTE solo a training para no tener un imbalanced de clases
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)

    # Entrenar
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Métricas
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= THRESHOLD).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    dropout_pct = y.mean() * 100

    # SHAP values — obtener feature más influyente
    explainer = shap.LinearExplainer(model, X_train)
    shap_values = explainer(X_test)
    mean_shap = pd.Series(
        np.abs(shap_values.values).mean(axis=0),
        index=X.columns
    )
    top_feature = mean_shap.idxmax()
    top_shap = mean_shap.max()

    return {
        'n': len(cluster_df),
        'dropout_pct': dropout_pct,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'top_feature': top_feature,
        'top_shap': top_shap,
        'model': model
    }


def print_results(results, label):
    print(f"\n{'='*75}")
    print(f"{label} — Métricas Logistic Regression por Cluster")
    print(f"{'='*75}")
    print(f"{'Cluster':<10} {'n':>7}  {'Dropout%':>8}  {'Precision':>9}  {'Recall':>6}  {'F1':>6}")
    print(f"{'-'*75}")
    for i, r in enumerate(results):
        print(f"C{i:<9} {r['n']:>7,}  {r['dropout_pct']:>7.1f}%  "
              f"{r['precision']:>9.3f}  {r['recall']:>6.3f}  {r['f1']:>6.3f}")

def print_shap_summary(results_dict):
    print(f"\n{'='*72}")
    print(f"Variable MÁS INFLUYENTE (SHAP) por Cluster")
    print(f"{'='*72}")
    for label, results in results_dict.items():
        print(f"{label}:")
        for i, r in enumerate(results):
            print(f"  C{i:<35} → {r['top_feature']:<25} (|SHAP|={r['top_shap']:.4f})")


Se obtienen todos los datos y se entrenan todos los clusters.

In [93]:
clusters_pre = load_clusters('../data/v5_dataset_pre_tec21_clusters.csv')
clusters_tec = load_clusters('../data/v5_dataset_tec21_clusters.csv')

# --- Entrenar modelos para cada cluster ---
results_pre, results_tec = [], []

for cluster_id, cluster_df in enumerate(clusters_pre):
    results_pre.append(train_cluster_model(cluster_df, cluster_id))

for cluster_id, cluster_df in enumerate(clusters_tec):
    results_tec.append(train_cluster_model(cluster_df, cluster_id))

================== Cargando:../data/v5_dataset_pre_tec21_clusters.csv
Cluster 0: 5694 filas
Cluster 1: 18392 filas
Cluster 2: 2586 filas
Cluster 3: 24270 filas
Cluster 4: 1667 filas
Cluster 5: 401 filas
Total de clusters creados: 6
================== Cargando:../data/v5_dataset_tec21_clusters.csv
Cluster 0: 7510 filas
Cluster 1: 8337 filas
Cluster 2: 3997 filas
Cluster 3: 938 filas
Cluster 4: 2894 filas
Cluster 5: 831 filas
Total de clusters creados: 6
Cluster 0 — columnas constantes dropeadas: ['is_foreign', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'online.test']
Cluster 1 — columnas constantes dropeadas: ['is_foreign', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'online.test']
Cluster 2 — columnas constantes dropeadas: ['is_foreign', 'first.generation.yes', 'has_extracurriculars', 'online.test']
Cluster 3 — columnas constantes dropeadas: ['is_foreign', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'online.

#### Impresion de resultados

In [94]:
print_results(results_pre, 'Pre-TEC21')
print_results(results_tec, 'TEC21')
print_shap_summary({'Pre-TEC21': results_pre, 'TEC21': results_tec})


Pre-TEC21 — Métricas Logistic Regression por Cluster
Cluster          n  Dropout%  Precision  Recall      F1
---------------------------------------------------------------------------
C0           5,694      4.8%      0.083   0.673   0.148
C1          18,392      6.2%      0.086   0.520   0.147
C2           2,586     10.4%      0.176   0.759   0.286
C3          24,270     11.1%      0.153   0.512   0.236
C4           1,667     14.2%      0.184   0.489   0.267
C5             401     20.2%      0.282   0.688   0.400

TEC21 — Métricas Logistic Regression por Cluster
Cluster          n  Dropout%  Precision  Recall      F1
---------------------------------------------------------------------------
C0           7,510      4.3%      0.049   0.538   0.090
C1           8,337      9.3%      0.121   0.587   0.200
C2           3,997     10.5%      0.137   0.667   0.227
C3             938     10.7%      0.116   0.400   0.180
C4           2,894     12.1%      0.171   0.600   0.266
C5             8